# Projet 2 —  Prédiction du risque d’accident de vélo

**Membres du groupe :**
- ABEGUEDIE Rosane Yvanne Boyama  
- BARRY Mamoudou  
- DAMIENS Elodie  
- GUILLEMINOT Paul  
- TOUAMI Zakaria

**Groupe :** D  
**Jeu de données :** `accidentsVelo.csv`  
**Objectif :** comprendre les facteurs associés à 

## 1. Problématique et logique d'analyse

...

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from scipy.stats import chi2_contingency, mannwhitneyu

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay,
    precision_recall_curve
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier

# XGBoost est utilisé seulement s'il est disponible
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

# Style graphique
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Chargement des données

On commence par charger le fichier CSV puis par vérifier sa structure générale.  
Cette étape est indispensable car elle permet de répondre aux premières questions :

- combien y a-t-il d'observations ?
- combien y a-t-il de variables ?
- quels sont les types de colonnes ?
- la cible est-elle équilibrée ou non ?

### But et utilité de l'étape

**But :** charger les données dans un format exploitable et vérifier rapidement leur structure (taille, types, valeurs manquantes).

**Utilité :** valider qu'on travaille sur le bon périmètre avant toute interprétation ou modélisation.


In [7]:
data = pd.read_csv("accidentsVelo.csv", sep=",")
display(data.head())

C:\Users\Elodi\AppData\Local\Temp\ipykernel_12024\3060811873.py:1: DtypeWarning: Columns (8,9,17,20,21,30) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("accidentsVelo.csv", sep=",")


,Num_Acc,date,an,mois,jour,hrmn,dep,com,lat,long,agg,int,col,lum,atm,catr,circ,nbv,prof,plan,lartpc,larrout,surf,infra,situ,grav,sexe,age,trajet,secuexist,equipement,obs,obsm,choc,manv,vehiculeid,typevehicules,manoeuvehicules,numVehicules
0,200500000030,2005-01-13,2005,janvier,jeudi,19:45,62,62331,50.300,2.840,2,1,3.000,5,1.000,3,2.000,0,-1.000,1.000,0,50,1.000,0.000,1.000,4,1,58.000,5.000,3,0,0.000,2.000,8.000,11.000,200500000030B02,18,17,1.000
1,200500000034,2005-01-19,2005,janvier,mercredi,10:45,62,62022,0.000,0.000,1,1,1.000,1,7.000,3,2.000,0,1.000,3.000,0,50,1.000,0.000,1.000,3,1,20.000,5.000,3,0,0.000,2.000,1.000,1.000,200500000034B02,10,15,1.000
2,200500000078,2005-01-26,2005,janvier,mercredi,13:15,02,02173,0.000,0.000,1,9,3.000,1,1.000,3,2.000,2,2.000,1.000,0,0,1.000,0.000,1.000,4,1,71.000,5.000,2,2,0.000,2.000,1.000,1.000,200500000078B02,7,15,1.000
3,200500000093,2005-01-03,2005,janvier,lundi,13:30,02,02810,49.255,3.094,2,1,1.000,1,1.000,3,2.000,0,1.000,2.000,0,52,1.000,0.000,1.000,3,2,51.000,4.000,3,0,0.000,2.000,3.000,21.000,200500000093B02,7,21,1.000
4,200500000170,2005-01-29,2005,janvier,samedi,18:30,76,76196,0.000,0.000,1,1,2.000,3,1.000,3,2.000,2,1.000,1.000,0,50,1.000,0.000,1.000,4,1,74.000,5.000,1,9,0.000,2.000,4.000,2.000,200500000170A01,10,2,1.000


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80022 entries, 0 to 80021
Data columns (total 39 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Num_Acc          80022 non-null  int64  
 1   date             80022 non-null  object 
 2   an               80022 non-null  int64  
 3   mois             80022 non-null  object 
 4   jour             80022 non-null  object 
 5   hrmn             80022 non-null  object 
 6   dep              80022 non-null  object 
 7   com              80022 non-null  object 
 8   lat              80022 non-null  object 
 9   long             79754 non-null  object 
 10  agg              80022 non-null  int64  
 11  int              80022 non-null  int64  
 12  col              80018 non-null  float64
 13  lum              80022 non-null  int64  
 14  atm              80019 non-null  float64
 15  catr             80022 non-null  int64  
 16  circ             79879 non-null  float64
 17  nbv         